# 10 · Hibrit model: füzyon yöntemleri ve üye ablasyonu

**GPU gerekmez** (kayıtlı doğrulama olasılıkları birleştirilir, birkaç dakika). Üyeler: Orig (özgün), DeiT, ConvNeXt, ResNet50 (`__opt` modelleri, her seed için kendi seed'iyle). Ağırlıklar ve meta-öğrenici macro F1 ölçütüyle seçilir (proje kuralı).

- 4 füzyon yöntemi: `mean` (eşit ağırlık), `weighted` (ağırlıklar doğrulamada), `stack` (lojistik regresyon meta-öğrenici), `mean_ord` (ortalama + sıralı karar kuralı)
- 11 üye kümesi (2, 3 ve 4 üyeli tüm kombinasyonlar) = üye ablasyonu
- Öğrenilen yöntemlerde doğrulama sonuçları 5 katlı çapraz doğrulama (katman dışı) ile hesaplanır, iyimser olmaz
- Yalnızca **validation**; test kullanılmaz

In [ ]:
import os, sys, glob, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # ---- Kaggle: kod ve önceki çıktılar /kaggle/input altındaki veri setlerinden gelir (salt okunur) ----
    hits = glob.glob('/kaggle/input/**/src/config.py', recursive=True)
    assert hits, 'Kod veri seti bulunamadı: sağ panelden Add Input ile yzproje-code veri setini ekleyin.'
    code_root = os.path.dirname(os.path.dirname(hits[0]))
    PROJECT = '/kaggle/working/YZProje'
    shutil.copytree(code_root, PROJECT, dirs_exist_ok=True)          # kod her oturumda tazelenir
    os.makedirs(PROJECT + '/outputs', exist_ok=True)
    out_hits = glob.glob('/kaggle/input/**/splits/all_files.csv', recursive=True)
    assert out_hits, 'Çıktı veri seti bulunamadı: Add Input ile yzproje-outputs veri setini ekleyin.'
    out_src = os.path.dirname(os.path.dirname(out_hits[0]))
    # yalnızca çalışma klasöründe OLMAYAN dosyaları kopyala (oturum içi yeni sonuçlar ezilmesin)
    shutil.copytree(out_src, PROJECT + '/outputs', dirs_exist_ok=True,
                    copy_function=lambda s, d: None if os.path.exists(d) else shutil.copy2(s, d))
    os.makedirs('/kaggle/temp/data', exist_ok=True)
    os.environ['KNEE_DATA_ROOT'] = '/kaggle/temp/data'                # ham veri: geçici disk (çıktıya sayılmaz)
else:
    # ---- Colab: Drive ----
    from google.colab import drive
    drive.mount('/content/drive')
    # Projeyi Drive'da bul (iç içe klasör olsa bile); zip açılmamışsa aç
    hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        zips = glob.glob('/content/drive/MyDrive/**/YZProje*.zip', recursive=True)
        if zips:
            zipfile.ZipFile(zips[0]).extractall('/content/drive/MyDrive/')
            hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        raise FileNotFoundError('Drive içinde src/config.py bulunamadı.')
    roots = [os.path.dirname(os.path.dirname(h)) for h in hits]
    # Birden fazla kopya varsa, veri listeleri (outputs/splits) olan klasör tercih edilir
    with_out = [r for r in roots if os.path.exists(f'{r}/outputs/splits/all_files.csv')]
    PROJECT = sorted(with_out or roots, key=len)[0]
    os.environ['KNEE_DATA_ROOT'] = '/content/data'

os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = f'{PROJECT}/outputs'
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| Proje klasörü:', PROJECT)
print(sorted(os.listdir(PROJECT + '/outputs')) if os.path.exists(PROJECT + '/outputs') else 'outputs yok (ilk çalıştırma)')

In [ ]:
from src import hybrid as H
import pandas as pd
summ = H.run_all(seeds=(42, 123, 2026))
show = ['method','members','macro_f1_mean','macro_f1_std','qwk_mean','mae_grade_mean','distant_error_rate_mean','accuracy_mean']
display(summ[show].head(15).round(4))

## Üye ablasyonu (yalnızca parametresiz `mean` yöntemi, yöntem etkisi karışmasın diye)

In [ ]:
display(summ[summ.method == 'mean'].sort_values('macro_f1_mean', ascending=False)[show].round(4))
print(H.choose_final(summ))

`outputs/hybrid_val_summary.csv`, `outputs/hybrid_val_runs.csv` ve `outputs/hybrid_choice.json` dosyalarını paylaşın.